# MCTS — 面试版

**难度：** Hard

**面试要点：** 纯 Python 实现，不用 torch 张量操作

In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def mcts_step(values, visit_counts, parent_visits, c_puct=1.414):
    # ---- 面试版: 用纯 Python 列表操作 ----
    # 转为 list 方便逐元素操作
    vals = values.clone().tolist()
    visits = visit_counts.clone().tolist()
    N = len(vals)

    # ---- Step 1: 手写 UCB1 计算 ----
    ucb_scores = []
    log_parent = math.log(parent_visits + 1)
    for i in range(N):
        # 利用项: 当前价值估计
        exploit = vals[i]
        # 探索项: c_puct * sqrt(log(N_parent+1) / (n_i+1))
        # 访问次数为 0 的节点，探索项最大，优先被探索
        explore = c_puct * math.sqrt(log_parent / (visits[i] + 1))
        ucb_scores.append(exploit + explore)

    # ---- Step 2: 手写 argmax ----
    # 不用 torch.argmax，用 Python 的 max + index
    selected_idx = max(range(N), key=lambda i: ucb_scores[i])

    # ---- Step 3: 随机 Rollout ----
    rollout_value = torch.rand(1).item()

    # ---- Step 4: 滑动均值更新 ----
    # 增量公式: new_avg = (old_avg * count + new_val) / (count + 1)
    n = visits[selected_idx]
    vals[selected_idx] = (vals[selected_idx] * n + rollout_value) / (n + 1)
    visits[selected_idx] += 1

    # 转回张量
    return selected_idx, torch.tensor(vals), torch.tensor(visits, dtype=torch.long)

In [ ]:
torch.manual_seed(42)
num_children = 5
values = torch.zeros(num_children)
visits = torch.zeros(num_children, dtype=torch.long)
parent_visits = 0

for step in range(5):
    idx, values, visits = mcts_step(values, visits, parent_visits)
    parent_visits += 1
    print(f"Step {step+1}: selected={idx}, visits={visits.tolist()}")

In [ ]:
from torch_judge import check
check("mcts_search")

## 📝 核心思路总结

1. **纯 Python 实现**：for 循环 + math.sqrt + list 操作
2. **手写 argmax**：`max(range(N), key=lambda i: ucb_scores[i])`
3. **增量均值**：避免存储所有历史值，O(1) 更新